# ArchAgent — Results Notebook

Charts the real run on **`andela/buggy-python`** (model `claude-sonnet-4-6`): node
centrality from the knowledge graph, and the baseline-vs-graph-guided token study.
Data is read from the committed deliverables, so re-running reproduces the figures.

Run from the `notebooks/` directory: `uv run jupyter lab` (or execute top-to-bottom).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from arch_agent.services.graph_loader import GraphLoader
from arch_agent.services.metrics import MetricsCalculator

graph = GraphLoader().load(Path("../artifacts/graph.json"))
metrics = MetricsCalculator(graph)
centrality = metrics.centrality()
fan_in, fan_out = metrics.fan_in(), metrics.fan_out()
print(f"{len(graph.nodes)} nodes, {len(graph.edges)} edges")

## 1. Node centrality (structural hubs)

In [ ]:
ranked = sorted(centrality, key=lambda n: centrality[n], reverse=True)
labels = [n.replace("snippets_", "") for n in ranked]
values = [centrality[n] for n in ranked]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(labels, values, color="#4C72B0")
ax.axhline(0.30, color="crimson", linestyle="--", label="God-Node threshold (0.30)")
ax.set_ylabel("degree centrality")
ax.set_title("Node centrality — buggy-python")
ax.tick_params(axis="x", rotation=60)
ax.legend()
fig.tight_layout()
fig.savefig("centrality.png", dpi=120)
plt.show()

## 2. Token efficiency — baseline vs graph-guided

Measured on the same debugging task. Savings:
$$\text{savings} = \frac{T_{\text{base}} - T_{\text{guided}}}{T_{\text{base}}}\times 100\%$$

In [ ]:
# measured token usage from reports/token_efficiency.md
baseline = {"input": 746, "output": 1230}
guided = {"input": 308, "output": 587}
base_total = sum(baseline.values())
guided_total = sum(guided.values())
savings = (base_total - guided_total) / base_total * 100

groups = ["input", "output", "total"]
base_vals = [baseline["input"], baseline["output"], base_total]
guided_vals = [guided["input"], guided["output"], guided_total]
x = range(len(groups))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([i - 0.2 for i in x], base_vals, width=0.4, label="baseline (raw code)", color="#C44E52")
ax.bar([i + 0.2 for i in x], guided_vals, width=0.4, label="graph-guided", color="#55A868")
ax.set_xticks(list(x))
ax.set_xticklabels(groups)
ax.set_ylabel("tokens")
ax.set_title(f"Token usage — graph-guided saves {savings:.1f}%")
ax.legend()
fig.tight_layout()
fig.savefig("token_savings.png", dpi=120)
plt.show()
print(f"savings: {savings:.1f}% (>= 40% KPI: {savings >= 40})")

## Takeaways

- **`snippets_io` is the central hub** (highest centrality) — the I/O layer everything routes through.
- **Graph-guided reasoning used ~50% fewer tokens** than dumping raw code, beating the 40% KPI, while still reaching the root cause with tests green.

See `reports/token_efficiency.md`, `reports/recommendations.md`, and `reports/before_after.md`.